In [1]:
import sys
import os
from pyspark.sql import SparkSession
from pyspark.sql.column import Column
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [2]:
APP_NAME = "Discharge App - Gold"

In [3]:
read_csv = lambda path: spark.read.option("header", "true").option("sep", ";").csv(path)
DATA_PATH = os.getenv("DATA_PATH")

## Check city nullity

In [4]:
schema = StructType([
    StructField("D_R", IntegerType(), True),
    StructField("ser_salud", IntegerType(), True),
    StructField("estab", IntegerType(), True),
    StructField("sexo", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("previ", IntegerType(), True),
    StructField("benef", IntegerType(), True),
    StructField("comuna", IntegerType(), True),
    StructField("fecha_egre", StringType(), True),
    StructField("sercle", StringType(), True),
    StructField("dias_estad", IntegerType(), True),
    StructField("diag1", StringType(), True),
    StructField("diag2", StringType(), True),
    StructField("cond_egr", IntegerType(), True),
    StructField("intervq", IntegerType(), True),
    StructField("region", IntegerType(), True),
    StructField("serv_res", StringType(), True),
])

raw_discharges = spark.read\
   .format('csv')\
   .option('header', 'true')\
   .option('sep', ';')\
   .schema(schema)\
   .load(f"{DATA_PATH}/raw/discharges/{{2001,2002,2003,2004,2006}}.csv")

raw_discharges.show()

+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
| D_R|ser_salud|estab|sexo|edad|previ|benef|comuna|fecha_egre|sercle|dias_estad|diag1|diag2|cond_egr|intervq|region|serv_res|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
|null|        1|  100|   2|  37|    1|    1|  1101|  7/8/2001|600-02|         1| O039|     |       1|      1|     1|       2|
|null|        1|  100|   1|  33|    1|    1|  1101| 8/27/2001|100-99|        16| C629|     |       1|      2|     1|       2|
|null|        1|  100|   1|   6|    1|    1|  1101| 5/24/2001|500-99|         1| A09X|     |       1|      2|     1|       2|
|null|        1|  100|   2|  47|    1|    1|  1101|  4/4/2001|200-00|        25| K810|     |       1|      1|     1|       2|
|null|        1|  100|   1|   0|    4| null|  1101|12/24/2001|500-99|         2| S411| V439|       1|      2|     1|  

In [59]:
raw_discharges.filter(F.col("comuna") >= 99999).show()

+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
| D_R|ser_salud|estab|sexo|edad|previ|benef|comuna|fecha_egre|sercle|dias_estad|diag1|diag2|cond_egr|intervq|region|serv_res|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
|null|        1|  100|   2|  38|    1|    2| 99999|10/26/2001|600-01|         3| O809|     |       1|      2|    99|      99|
|null|        1|  100|   1|  55|    1|    2| 99999| 5/14/2001|300-00|         7| S069| V439|       1|      2|    99|      99|
|null|        1|  100|   2|  35|    1|    2| 99999| 7/21/2001|600-01|         3| O021|     |       1|      1|    99|      99|
|null|        1|  100|   1|  36|    1|    2| 99999| 6/28/2001|300-00|        18| S021| W194|       1|      2|    99|      99|
|null|        1|  100|   1|  49|    1|    2| 99999|  5/8/2001|200-00|         2| N433|     |       1|      1|    99|  

## Dimension

In [4]:
dim_city_df = read_csv(f"{DATA_PATH}/raw/dimensions/dim_city.csv")\
    .select(
        F.col("id").alias("city_id"),
        F.col("city_cut"),
        F.col("valid_from").cast("integer"),
        F.col("valid_to").cast("integer")
    )

dim_city_df.show()

+-------+--------+----------+--------+
|city_id|city_cut|valid_from|valid_to|
+-------+--------+----------+--------+
| 999999|  999999|      1111|  999999|
|      1|     570|      1990|    1999|
|      2|   11201|      2000|    2018|
|      3|     324|      1990|    1999|
|      4|    5602|      2000|    2018|
|      5|     663|      1990|    1999|
|      6|   13502|      2000|    2018|
|      8|    8314|      2004|    2018|
|      9|     248|      1990|    1999|
|     10|    3302|      2000|    2018|
|     12|    1107|      2004|    2018|
|     13|     546|      1990|    1999|
|     14|   10202|      2000|    2007|
|     15|   10202|      2008|    2018|
|     16|     253|      1990|    1999|
|     17|    4103|      2000|    2018|
|     18|     475|      1990|    1999|
|     19|    9201|      2000|    2018|
|     20|     230|      1990|    1999|
|     21|    2101|      2000|    2018|
+-------+--------+----------+--------+
only showing top 20 rows



In [5]:
dim_city_df.count()

738

In [7]:
dim_city_df.filter((F.col("city_cut").isNull()) | (F.col("city_cut") == "")).show()

+-------+--------+----------+--------+
|city_id|city_cut|valid_from|valid_to|
+-------+--------+----------+--------+
+-------+--------+----------+--------+



In [6]:
invalid_cuts_df = read_csv(f"{DATA_PATH}/raw/dimensions/city.invalid_cuts.lookup.csv")
invalid_cuts_df.show()

+------+
|   CUT|
+------+
|999999|
| 20000|
|    51|
|    54|
|   591|
| 88888|
|  2000|
+------+



In [9]:
invalid_cuts_df.count()

7

In [10]:
invalid_cuts_df.filter((F.col("CUT").isNull()) | (F.col("CUT") == "")).show()

+---+
|CUT|
+---+
+---+



## Fact table (Silver cleaned)

In [71]:
windowSpec = Window.orderBy(F.monotonically_increasing_id())

discharges_df = spark\
    .read\
    .parquet(f"{DATA_PATH}/silver/cleaned_discharges")\
    .withColumn("id", F.row_number().over(windowSpec))\
    .select("id", "year", "age", "gender", "health_insurance", "city", "healthcare_service", "cause_of_death_1", "cause_of_death_2")#\
    #.filter((F.col("city") == 5507) & (F.col("year") == 2010))

discharges_df.show()

26/05/11 15:43:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:43:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:43:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 278:=================================================>       (7 + 1) / 8]

+---+----+---+------+----------------+----+------------------+----------------+----------------+
| id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|
+---+----+---+------+----------------+----+------------------+----------------+----------------+
|  1|2001| 37|     2|               1|1101|                 1|            O039|             N/E|
|  2|2001| 55|     1|               1|5108|                 7|            M139|             N/E|
|  3|2001| 33|     1|               1|1101|                 1|            C629|             N/E|
|  4|2001| 30|     2|               1|5108|                 7|            O809|             N/E|
|  5|2001|  6|     1|               1|1101|                 1|            A09X|             N/E|
|  6|2001| 23|     2|               1|5108|                 7|            O809|             N/E|
|  7|2001| 47|     2|               1|1101|                 1|            K810|             N/E|
|  8|2001| 17|     2|         

In [44]:
discharges_df.filter((F.col("city").isNull()) | (F.col("city") == "") | (F.col("city") == 999999) | (F.col("city") == 99999)).show()

26/05/11 02:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 02:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 02:54:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 02:55:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 02:55:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 162:>                                                        (0 + 1) / 1]

+-----+----+---+------+----------------+------+------------------+----------------+----------------+
|   id|year|age|gender|health_insurance|  city|healthcare_service|cause_of_death_1|cause_of_death_2|
+-----+----+---+------+----------------+------+------------------+----------------+----------------+
|27549|2001| 38|     2|               1|999999|                 1|            O809|             N/E|
|27551|2001| 55|     1|               1|999999|                 1|            S069|            V439|
|27553|2001| 35|     2|               1|999999|                 1|            O021|             N/E|
|27555|2001| 36|     1|               1|999999|                 1|            S021|            W194|
|27557|2001| 49|     1|               1|999999|                 1|            N433|             N/E|
|27559|2001| 23|     2|               1|999999|                 1|            O069|             N/E|
|27561|2001| 35|     2|               1|999999|                 1|            O021|        

In [60]:
discharges_df.filter(F.col("city") == 99999).show()

26/05/11 03:21:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:21:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:21:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:21:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:21:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 235:>                                                        (0 + 1) / 1]

+---+----+---+------+----------------+----+------------------+----------------+----------------+
| id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|
+---+----+---+------+----------------+----+------------------+----------------+----------------+
+---+----+---+------+----------------+----+------------------+----------------+----------------+



## Join

### Step 1

In [72]:
step_1 = discharges_df\
    .join(
        F.broadcast(invalid_cuts_df),
        F.col("city") == F.col("CUT"),
        how="left"
    )

step_1.show()

26/05/11 15:47:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:47:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:47:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:47:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 281:=================================================>       (7 + 1) / 8]

+---+----+---+------+----------------+----+------------------+----------------+----------------+----+
| id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2| CUT|
+---+----+---+------+----------------+----+------------------+----------------+----------------+----+
|  1|2001| 37|     2|               1|1101|                 1|            O039|             N/E|null|
|  2|2001| 55|     1|               1|5108|                 7|            M139|             N/E|null|
|  3|2001| 33|     1|               1|1101|                 1|            C629|             N/E|null|
|  4|2001| 30|     2|               1|5108|                 7|            O809|             N/E|null|
|  5|2001|  6|     1|               1|1101|                 1|            A09X|             N/E|null|
|  6|2001| 23|     2|               1|5108|                 7|            O809|             N/E|null|
|  7|2001| 47|     2|               1|1101|                 1|            K810|   

In [62]:
(step_1.filter(F.col("CUT").isNull()).count(), step_1.filter(F.col("CUT").isNotNull()).count())

(1317, 0)

In [63]:
step_1.filter(F.col("CUT").isNull()).show()

26/05/11 03:23:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:23:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:23:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:23:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:23:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 03:23:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 250

+--------+----+---+------+----------------+----+------------------+----------------+----------------+----+
|      id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2| CUT|
+--------+----+---+------+----------------+----+------------------+----------------+----------------+----+
|14405074|2010| 34|     2|               1|5507|                 7|            O809|             N/E|null|
|14406882|2010| 18|     1|               3|5507|                 9|            K358|             N/E|null|
|14410557|2010|  5|     2|               1|5507|                 7|            R590|             N/E|null|
|14410561|2010| 92|     2|               1|5507|                 7|            J189|             N/E|null|
|14414440|2010| 92|     2|               1|5507|                 7|            I509|             N/E|null|
|14414441|2010| 57|     1|               1|5507|                 7|            I200|             N/E|null|
|14419037|2010| 86|     1|           

In [32]:
step_1.filter(F.col("CUT").isNotNull()).show()

+---+----+---+------+----------------+----+------------------+----------------+----------------+---+
| id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|CUT|
+---+----+---+------+----------------+----+------------------+----------------+----------------+---+
+---+----+---+------+----------------+----+------------------+----------------+----------------+---+



26/05/10 00:50:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:50:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 103:=================================================>       (7 + 1) / 8]

### Step 2

In [73]:
step_2 = step_1.join(
    F.broadcast(dim_city_df),
    (F.col("year") >= F.col("valid_from")) & 
    (F.col("year") <= F.col("valid_to")) &
    (F.coalesce(F.when(F.col("CUT").isNotNull(), F.lit(999999)), F.col("city")) == F.col("city_cut")),
    how="left"
)

step_2.show()

26/05/11 15:49:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:49:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:49:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:49:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:49:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:49:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 285

+---+----+---+------+----------------+----+------------------+----------------+----------------+----+-------+--------+----------+--------+
| id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2| CUT|city_id|city_cut|valid_from|valid_to|
+---+----+---+------+----------------+----+------------------+----------------+----------------+----+-------+--------+----------+--------+
|  1|2001| 37|     2|               1|1101|                 1|            O039|             N/E|null|    248|    1101|      2000|    2018|
|  2|2001| 55|     1|               1|5108|                 7|            M139|             N/E|null|    722|    5108|      2000|    2009|
|  3|2001| 33|     1|               1|1101|                 1|            C629|             N/E|null|    248|    1101|      2000|    2018|
|  4|2001| 30|     2|               1|5108|                 7|            O809|             N/E|null|    722|    5108|      2000|    2009|
|  5|2001|  6|     1|      

In [74]:
step_2\
    .filter(F.col("city_id").isNull())\
    .groupBy("year", "city")\
    .count()\
    .show()

[Stage 289:=================================================>       (7 + 1) / 8]

+----+----+-----+
|year|city|count|
+----+----+-----+
+----+----+-----+



26/05/11 16:15:42 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-11422bb8-832f-49c2-86e0-94d062eba5f9. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-11422bb8-832f-49c2-86e0-94d062eba5f9
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:177)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:113)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1231)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptimized.scala:36)
	at scala.collection.IndexedSeqOptimized.foreach$(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofRef.foreach

In [51]:
(step_2.filter(F.col("city_id").isNull()).count(), step_2.filter(F.col("city_id").isNotNull()).count())

(32783, 29256451)

In [34]:
step_2.filter(F.col("city_cut") == 999999).show()

26/05/10 00:51:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:51:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:51:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:51:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:51:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:51:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 0

+---+----+---+------+----------------+----+------------------+----------------+----------------+---+-------+--------+----------+--------+
| id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2|CUT|city_id|city_cut|valid_from|valid_to|
+---+----+---+------+----------------+----+------------------+----------------+----------------+---+-------+--------+----------+--------+
+---+----+---+------+----------------+----+------------------+----------------+----------------+---+-------+--------+----------+--------+



In [37]:
step_2.filter((F.col("city_id").isNull()) | (F.col("city_id") == "") | (F.col("city_id") == 999999)).show()

26/05/10 00:53:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:53:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:53:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:53:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:53:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:53:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 0

+--------+----+---+------+----------------+----+------------------+----------------+----------------+----+-------+--------+----------+--------+
|      id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2| CUT|city_id|city_cut|valid_from|valid_to|
+--------+----+---+------+----------------+----+------------------+----------------+----------------+----+-------+--------+----------+--------+
|14404970|2010| 61|     1|               1|5106|                 6|            C793|             N/E|null|   null|    null|      null|    null|
|14405011|2010| 82|     2|               1|5106|                 7|            A419|             N/E|null|   null|    null|      null|    null|
|14405023|2010| 22|     2|               1|5106|                 7|            O601|             N/E|null|   null|    null|      null|    null|
|14405029|2010| 32|     2|               1|5106|                 7|            O809|             N/E|null|   null|    null|      null|  

In [36]:
step_2.filter(F.col("city_id").isNull()).show()

26/05/10 00:52:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:52:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:52:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:52:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:52:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 00:52:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 0

+--------+----+---+------+----------------+----+------------------+----------------+----------------+----+-------+--------+----------+--------+
|      id|year|age|gender|health_insurance|city|healthcare_service|cause_of_death_1|cause_of_death_2| CUT|city_id|city_cut|valid_from|valid_to|
+--------+----+---+------+----------------+----+------------------+----------------+----------------+----+-------+--------+----------+--------+
|14404970|2010| 61|     1|               1|5106|                 6|            C793|             N/E|null|   null|    null|      null|    null|
|14405011|2010| 82|     2|               1|5106|                 7|            A419|             N/E|null|   null|    null|      null|    null|
|14405023|2010| 22|     2|               1|5106|                 7|            O601|             N/E|null|   null|    null|      null|    null|
|14405029|2010| 32|     2|               1|5106|                 7|            O809|             N/E|null|   null|    null|      null|  

In [19]:
step_2.filter(F.col("city_id").isNull()).count()

38759

In [22]:
step_2\
    .filter(F.col("city_id").isNull())\
    .groupBy("year")\
    .count()\
    .sort(F.col("year").asc())\
    .show()

[Stage 60:==================================================>       (7 + 1) / 8]

+----+-----+
|year|count|
+----+-----+
|2004|   14|
|2005|  349|
|2006|  215|
|2007|  468|
|2008|  315|
|2009|  587|
|2010|32783|
|2011|  172|
|2012|  127|
|2013|  302|
|2014|  173|
|2015|  217|
|2016|  302|
|2017|  718|
|2018| 2017|
+----+-----+



## Debug

In [59]:
debug_df = discharges_df\
    .filter(
        (F.col("year") == 2004) &
        (F.col("city") == 20000)
    ).union(
        discharges_df\
            .filter(
                (F.col("year") == 2004) &
                (F.col("city") != 20000)
            )\
            .limit(14)
    )

debug_df.show(30)

26/05/06 02:23:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:23:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:23:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:23:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:23:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:23:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 0

+-------+----+---+------+----------------+-----+------------------+----------------+----------------+
|     id|year|age|gender|health_insurance| city|healthcare_service|cause_of_death_1|cause_of_death_2|
+-------+----+---+------+----------------+-----+------------------+----------------+----------------+
|4634784|2004| 62|     1|               1|20000|                 1|            N189|             N/E|
|5333323|2004| 35|     1|               6|20000|                12|            N201|             N/E|
|6133113|2004| 35|     1|               7|20000|                24|            R31X|             N/E|
|6133117|2004| 20|     1|               6|20000|                24|            L024|             N/E|
|6138363|2004| 68|     2|               6|20000|                24|            T029|            V499|
|6138364|2004| 36|     1|               6|20000|                24|            A09X|             N/E|
|6138365|2004| 38|     1|               6|20000|                24|            A09

In [60]:
debug_df = debug_df\
    .join(
        F.broadcast(invalid_cuts_df),
        F.col("city") == F.col("CUT"),
        how="left"
    )

debug_df.show(30)

26/05/06 02:24:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:24:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:24:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:24:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:24:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:24:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 0

+-------+----+---+------+----------------+-----+------------------+----------------+----------------+-----+
|     id|year|age|gender|health_insurance| city|healthcare_service|cause_of_death_1|cause_of_death_2|  CUT|
+-------+----+---+------+----------------+-----+------------------+----------------+----------------+-----+
|4634784|2004| 62|     1|               1|20000|                 1|            N189|             N/E|20000|
|5333323|2004| 35|     1|               6|20000|                12|            N201|             N/E|20000|
|6133113|2004| 35|     1|               7|20000|                24|            R31X|             N/E|20000|
|6133117|2004| 20|     1|               6|20000|                24|            L024|             N/E|20000|
|6138363|2004| 68|     2|               6|20000|                24|            T029|            V499|20000|
|6138364|2004| 36|     1|               6|20000|                24|            A09X|             N/E|20000|
|6138365|2004| 38|     1|   

In [61]:
debug_df = debug_df\
    .join(
        F.broadcast(dim_city_df),
        (F.col("year") >= F.col("valid_from")) & 
        (F.col("year") <= F.col("valid_to")) &
        (F.coalesce(F.when(F.col("CUT").isNotNull(), F.lit(999999)), F.col("city")) == F.col("city_cut")),
        how="left"
    )

debug_df.show(30)

26/05/06 02:25:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:25:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:25:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:25:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:25:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 02:25:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/06 0

+-------+----+---+------+----------------+-----+------------------+----------------+----------------+-----+-------+--------+----------+--------+
|     id|year|age|gender|health_insurance| city|healthcare_service|cause_of_death_1|cause_of_death_2|  CUT|city_id|city_cut|valid_from|valid_to|
+-------+----+---+------+----------------+-----+------------------+----------------+----------------+-----+-------+--------+----------+--------+
|4634784|2004| 62|     1|               1|20000|                 1|            N189|             N/E|20000| 999999|  999999|      1111|  999999|
|5333323|2004| 35|     1|               6|20000|                12|            N201|             N/E|20000| 999999|  999999|      1111|  999999|
|6133113|2004| 35|     1|               7|20000|                24|            R31X|             N/E|20000| 999999|  999999|      1111|  999999|
|6133117|2004| 20|     1|               6|20000|                24|            L024|             N/E|20000| 999999|  999999|      

In [62]:
debug_df\
    .select(
        F.col("health_insurance"),
        F.col("year"),
        F.col("gender"),
        F.col("age"),
        F.col("city_id"),
        F.col("healthcare_service"),
        F.col("cause_of_death_1"),
        F.col("cause_of_death_2")      
    )\
    .show(30)

+----------------+----+------+---+-------+------------------+----------------+----------------+
|health_insurance|year|gender|age|city_id|healthcare_service|cause_of_death_1|cause_of_death_2|
+----------------+----+------+---+-------+------------------+----------------+----------------+
|               1|2004|     1| 62| 999999|                 1|            N189|             N/E|
|               6|2004|     1| 35| 999999|                12|            N201|             N/E|
|               7|2004|     1| 35| 999999|                24|            R31X|             N/E|
|               6|2004|     1| 20| 999999|                24|            L024|             N/E|
|               6|2004|     2| 68| 999999|                24|            T029|            V499|
|               6|2004|     1| 36| 999999|                24|            A09X|             N/E|
|               6|2004|     1| 38| 999999|                24|            A09X|             N/E|
|               6|2004|     1| 43| 99999

## Fact table (gold)

In [69]:
windowSpec = Window.orderBy(F.monotonically_increasing_id())

gold_df = spark\
    .read\
    .option("header", "true") \
    .csv(f"{DATA_PATH}/gold/discharges/part-*")\
    .withColumn("id", F.row_number().over(windowSpec))\
    .select(
        "id", 
        "health_insurance_id",
        "year_id", 
        "gender_id", 
        "age_id", 
        "city_id", 
        "healthcare_service_id", 
        "main_cause_of_death_id",
        "external_cause_of_death_id"
    )

gold_df.show()

26/05/11 15:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 15:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 272:==============>                                          (2 + 6) / 8]

+---+-------------------+-------+---------+------+-------+---------------------+----------------------+--------------------------+
| id|health_insurance_id|year_id|gender_id|age_id|city_id|healthcare_service_id|main_cause_of_death_id|external_cause_of_death_id|
+---+-------------------+-------+---------+------+-------+---------------------+----------------------+--------------------------+
|  1|                  1|     12|        2|    46|    248|                    7|                  5635|                    999999|
|  2|                  1|     12|        1|    28|    722|                   32|                  4699|                    999999|
|  3|                  1|     12|        1|    50|    248|                    7|                  1055|                    999999|
|  4|                  1|     12|        2|    53|    722|                   32|                  5954|                    999999|
|  5|                  1|     12|        1|    77|    248|                    7|   

In [70]:
gold_df.filter(F.col("city_id").isNull()).count()

0

## Fact table (Raw)

In [23]:
schema = StructType([
    StructField("D_R", IntegerType(), True),
    StructField("ser_salud", IntegerType(), True),
    StructField("estab", IntegerType(), True),
    StructField("sexo", IntegerType(), True),
    StructField("edad", IntegerType(), True),
    StructField("previ", IntegerType(), True),
    StructField("benef", IntegerType(), True),
    StructField("comuna", IntegerType(), True),
    StructField("fecha_egre", StringType(), True),
    StructField("sercle", StringType(), True),
    StructField("dias_estad", IntegerType(), True),
    StructField("diag1", StringType(), True),
    StructField("diag2", StringType(), True),
    StructField("cond_egr", IntegerType(), True),
    StructField("intervq", IntegerType(), True),
    StructField("region", IntegerType(), True),
    StructField("serv_res", StringType(), True),
])

raw_discharges = spark.read\
    .format('csv')\
    .option('header', 'true')\
    .option('sep', ';')\
    .schema(schema)\
    .load(f"{DATA_PATH}/raw/discharges/{{2001,2002,2003,2004,2006}}.csv")

raw_discharges.show()

+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
| D_R|ser_salud|estab|sexo|edad|previ|benef|comuna|fecha_egre|sercle|dias_estad|diag1|diag2|cond_egr|intervq|region|serv_res|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
|null|        1|  100|   2|  37|    1|    1|  1101|  7/8/2001|600-02|         1| O039|     |       1|      1|     1|       2|
|null|        1|  100|   1|  33|    1|    1|  1101| 8/27/2001|100-99|        16| C629|     |       1|      2|     1|       2|
|null|        1|  100|   1|   6|    1|    1|  1101| 5/24/2001|500-99|         1| A09X|     |       1|      2|     1|       2|
|null|        1|  100|   2|  47|    1|    1|  1101|  4/4/2001|200-00|        25| K810|     |       1|      1|     1|       2|
|null|        1|  100|   1|   0|    4| null|  1101|12/24/2001|500-99|         2| S411| V439|       1|      2|     1|  

In [37]:
df = raw_discharges\
    .filter(
        F.col("fecha_egre").endswith("2004") &
        (F.col("comuna") == 20000)
    )

df.show(100)

+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
| D_R|ser_salud|estab|sexo|edad|previ|benef|comuna|fecha_egre|sercle|dias_estad|diag1|diag2|cond_egr|intervq|region|serv_res|
+----+---------+-----+----+----+-----+-----+------+----------+------+----------+-----+-----+--------+-------+------+--------+
|null|        1|  100|   1|  62|    1|    1| 20000|06-11-2004|   110|        15| N189|     |       1|      2|    99|      99|
|null|       12|  212|   1|  35|    6| null| 20000|02-08-2004|   270|         1| N201|     |       1|      1|    99|      99|
|null|       24|  250|   1|  35|    7| null| 20000|09-01-2004|   999|         1| R31X|     |       1|      1|    99|      99|
|null|       24|  250|   1|  20|    6| null| 20000|01-04-2004|   999|         1| L024|     |       1|      1|    99|      99|
|null|       24|  260|   2|  68|    6| null| 20000|13-01-2004|   999|         5| T029| V499|       1|      2|    99|  

In [35]:
raw_discharges\
    .groupBy("comuna")\
    .count()\
    .sort(F.col("count").asc())\
    .show()

[Stage 88:=================>                                       (3 + 7) / 10]

+------+-----+
|comuna|count|
+------+-----+
| 12103|    7|
|   591|   10|
| 12303|   11|
| 20000|   14|
|    51|   19|
|    54|   27|
| 12102|   28|
| 12202|   31|
| 12302|   41|
|  2202|   49|
|  1202|   50|
|  1302|   59|
| 12104|   87|
| 12402|  102|
| 11303|  134|
|  2000|  159|
|  1103|  161|
| 11302|  163|
|  5104|  201|
|  1107|  242|
+------+-----+
only showing top 20 rows

